# Gradient boosting

LightGBM with early stopping, then CatBoost, on all features (`add_features`). The number of trees is chosen on
valid, so valid scores are slightly optimistic; the final check is on test.

In [2]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score

from fraud.data import load_train
from fraud.features import add_features
from fraud.split import time_split

In [3]:
df = add_features(load_train())
train, valid, _ = time_split(df)

NON_FEATURES = ['isFraud', 'TransactionID', 'TransactionDT']
X_train, y_train = train.drop(columns=NON_FEATURES), train['isFraud']
X_valid, y_valid = valid.drop(columns=NON_FEATURES),valid['isFraud']

In [4]:
MODEL_PARAMS = dict(
    n_estimators=3000,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.5,
    metric='average_precision',
    n_jobs=8,
    verbose=-1,
)

for seed in [42,1,2]:
    model = lgb.LGBMClassifier(**MODEL_PARAMS, random_state=seed)
    model.fit(X_train, y_train, eval_X=X_valid, eval_y=y_valid, callbacks=[lgb.early_stopping(100, verbose=False)])
    scores = model.predict_proba(X_valid)[:, 1]
    print(f"seed {seed:<3} trees {model.best_iteration_:<5} "
          f"ROC-AUC {roc_auc_score(y_valid, scores):.4f}   PR-AUC {average_precision_score(y_valid, scores):.4f}")

seed 42  trees 1893  ROC-AUC 0.9290   PR-AUC 0.6510
seed 1   trees 1390  ROC-AUC 0.9309   PR-AUC 0.6483
seed 2   trees 1195  ROC-AUC 0.9327   PR-AUC 0.6442


With early stopping on valid PR-AUC (patience 100, up to 3,000 trees), LightGBM picks 1,893, 1,390 and 1,195 trees
for seeds 42, 1, 2 and reaches PR-AUC 0.6510, 0.6483 and 0.6442 (mean 0.648, ROC-AUC 0.931), against 0.607 with
300 trees: 300 trees were too few. The best number of trees varies a lot between seeds while the score barely
does, so the valid curve is flat near the top.

In [5]:
from catboost import CatBoostClassifier

CAT_COLS = X_train.select_dtypes('category').columns.tolist()
X_train_cb, X_valid_cb = X_train.copy(), X_valid.copy()
X_train_cb[CAT_COLS] = (X_train_cb[CAT_COLS].astype("object").fillna("missing").astype(str))
X_valid_cb[CAT_COLS] = (X_valid_cb[CAT_COLS].astype("object").fillna("missing").astype(str))

for seed in [42, 1, 2]:
    model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.1,
        eval_metric="PRAUC",
        early_stopping_rounds=100,
        task_type="GPU",
        random_seed=seed,
        verbose=0,
    )
    model.fit(X_train_cb, y_train, cat_features=CAT_COLS, eval_set=(X_valid_cb, y_valid))
    scores = model.predict_proba(X_valid_cb)[:, 1]
    print(f"seed {seed:<3} trees {model.get_best_iteration():<5} "
          f"ROC-AUC {roc_auc_score(y_valid, scores):.4f}   PR-AUC {average_precision_score(y_valid, scores):.4f}")

Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


seed 42  trees 1743  ROC-AUC 0.9100   PR-AUC 0.5937


Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


seed 1   trees 1561  ROC-AUC 0.9088   PR-AUC 0.5921


Default metric period is 5 because PRAUC is/are not implemented for GPU
Metric PRAUC is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


seed 2   trees 2484  ROC-AUC 0.9136   PR-AUC 0.6000


CatBoost on the same features (categories as text, missing values as "missing"; GPU, learning rate 0.1, early
stopping on valid PR-AUC): PR-AUC 0.6049, 0.6066 and 0.5937 over seeds 42, 1, 2 (mean 0.602, ROC-AUC 0.912) with
2,079–2,436 trees. Every seed is below the worst LightGBM seed (0.644), so LightGBM stays the main model. CatBoost
was not tuned; on CPU it needs over 30 minutes per run, and GPU runs are not exactly repeatable even with the same
seed.

## Parameter search

Parameters are searched on train only: each trial fits on days 1–90 and is scored on days 91–120. Valid is used
once afterwards, to compare the best parameters found with the current ones.

In [6]:
import optuna
from fraud.split import SECONDS_PER_DAY

day = train['TransactionDT'] // SECONDS_PER_DAY
inner_fit = (day<91).to_numpy()

X_fit, y_fit = X_train[inner_fit], y_train[inner_fit]
X_tune, y_tune = X_train[~inner_fit], y_train[~inner_fit]

In [8]:
def objective(trial):
    params = dict(
        n_estimators=3000,
        learning_rate=0.05,
        num_leaves=trial.suggest_int("num_leaves", 15, 255, log=True),
        min_child_samples=trial.suggest_int("min_child_samples", 10, 300, log=True),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.2, 1.0),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        subsample_freq=1,
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 100, log=True),
        metric="average_precision",
        n_jobs=8,
        verbose=-1,
        random_state=42,
    )
    model = lgb.LGBMClassifier(**params)
    model.fit(X_fit, y_fit,eval_X=X_tune, eval_y=y_tune, callbacks=[lgb.early_stopping(100, verbose=False)])
    return model.best_score_['valid_0']['average_precision']

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=40)

[I 2026-11-10 21:35:08,773] A new study created in memory with name: no-name-e96c7fae-63c1-48b6-a5f2-f2b8d4e861ef
[I 2026-11-10 21:35:48,709] Trial 0 finished with value: 0.5999093448579479 and parameters: {'num_leaves': 42, 'min_child_samples': 253, 'colsample_bytree': 0.7855951534491241, 'subsample': 0.7993292420985183, 'reg_lambda': 0.006026889128682512}. Best is trial 0 with value: 0.5999093448579479.
[I 2026-11-10 21:36:27,453] Trial 1 finished with value: 0.5973879050364768 and parameters: {'num_leaves': 23, 'min_child_samples': 12, 'colsample_bytree': 0.8929409166199482, 'subsample': 0.8005575058716043, 'reg_lambda': 3.4702669886504163}. Best is trial 0 with value: 0.5999093448579479.
[I 2026-11-10 21:36:53,324] Trial 2 finished with value: 0.5761210853747354 and parameters: {'num_leaves': 15, 'min_child_samples': 271, 'colsample_bytree': 0.8659541126403374, 'subsample': 0.6061695553391381, 'reg_lambda': 0.008111941985431923}. Best is trial 0 with value: 0.5999093448579479.
[I 2

In [9]:
print(f"best PR-AUC on days 91-120: {study.best_value:.4f}")
study.best_params

best PR-AUC on days 91-120: 0.6461


{'num_leaves': 246,
 'min_child_samples': 24,
 'colsample_bytree': 0.500585871573167,
 'subsample': 0.8757727458357346,
 'reg_lambda': 0.07278911568099064}

40 trials (TPE, seed 42). The score rises with the number of leaves: trials with 15-42 leaves reach 0.58-0.60 on
days 91-120, and the top twelve all have 181-255 leaves (0.640-0.646). Small `min_child_samples` (10-24),
`subsample` around 0.9 and weak `reg_lambda` also do better. Best trial: 246 leaves, `min_child_samples` 24,
`colsample_bytree` 0.50, `subsample` 0.88, `reg_lambda` 0.073 (PR-AUC 0.646). The top is flat, and the number of
leaves reaches the end of the range, so even larger trees might help.

In [11]:
TUNED_PARAMS = MODEL_PARAMS | {
    "num_leaves": 246,
    "min_child_samples": 24,
    "colsample_bytree": 0.50,
    "subsample": 0.88,
    "reg_lambda": 0.073,
}

for seed in [42, 1, 2]:
    model = lgb.LGBMClassifier(**TUNED_PARAMS, random_state=seed)
    model.fit(X_train, y_train, eval_X=X_valid, eval_y=y_valid,
              callbacks=[lgb.early_stopping(100, verbose=False)])
    scores = model.predict_proba(X_valid)[:, 1]
    print(f"seed {seed:<3} trees {model.best_iteration_:<5} "
          f"ROC-AUC {roc_auc_score(y_valid, scores):.4f}   PR-AUC {average_precision_score(y_valid, scores):.4f}")

seed 42  trees 473   ROC-AUC 0.9351   PR-AUC 0.6540
seed 1   trees 353   ROC-AUC 0.9365   PR-AUC 0.6516
seed 2   trees 1855  ROC-AUC 0.9291   PR-AUC 0.6727


On valid, the tuned parameters give PR-AUC 0.6540, 0.6516 and 0.6727 over seeds 42, 1, 2 (mean 0.659, ROC-AUC
0.934) against 0.6510, 0.6483 and 0.6442 (mean 0.648) with the current ones: every seed beats the best current
seed, though seed 1 only by 0.0006. The tuned parameters are kept. Early stopping picks 473, 353 and 1,855 trees,
and the seed with the most trees scores highest, so with larger trees a patience of 100 may stop too early.